In [1]:
import json
import os

import numpy as np
import papermill as pm
from jaxcmr.helpers import find_project_root
from jaxcmr.summarize import pairwise_cv_differences, cv_winner_comparison_matrix

# Cross-Validation: Model Comparison

Runs leave-one-list-out CV for each model variant, then compares held-out NLL.

In [2]:
base_params = {
    "redo_cv": True,
    "max_subjects": 0,
    "data_tag": "TalmiEEG",
    "data_path": "data/TalmiEEG.h5",
    "trial_query": "data['subject'] > -1",
    "target_directory": "projects/TalmiEEG/results/",
    "component_paths": {
        "mfc_create_fn": "jaxcmr.components.linear_memory.init_mfc",
        "mcf_create_fn": "jaxcmr.components.linear_memory.init_mcf",
        "context_create_fn": "jaxcmr.components.context.init",
        "termination_policy_create_fn": "jaxcmr.components.termination.NoStopTermination",
    },
    "loss_fn_path": "jaxcmr.loss.set_permutation_likelihood.MemorySearchLikelihoodFnGenerator",
    "fit_alg_path": "jaxcmr.fitting.ScipyDE",
    "fold_field": "list",
    "cv_best_of": 1,
    "base_run_tag": "50_set_likelihood_fixed_term",
    "best_of": 3,
    "relative_tolerance": 0.001,
    "popsize": 15,
    "num_steps": 1000,
    "cross_rate": 0.9,
    "diff_w": 0.85,
}

In [3]:
varied_parameters = [
    # Emotion-only CMR (no LPP terms)
    {
        "model_name": "EEGEmotionOnly",
        "make_factory_path": "jaxcmr.models_eeg.eeg_cmr.make_factory",
        "parameters": {
            "fixed": {
                "allow_repeated_recalls": False,
                "modulate_emotion_by_primacy": False,
                "learn_after_context_update": False,
                "lpp_main_scale": 0.0,
                "lpp_main_threshold": 0.0,
                "lpp_inter_scale": 0.0,
                "lpp_inter_threshold": 0.0,
            },
            "free": {
                "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "shared_support": [2.220446049250313e-16, 100.0],
                "item_support": [2.220446049250313e-16, 100.0],
                "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
                "primacy_scale": [2.220446049250313e-16, 100.0],
                "primacy_decay": [2.220446049250313e-16, 100.0],
                "choice_sensitivity": [2.220446049250313e-16, 100.0],
                "emotion_scale": [2.220446049250313e-16, 10.0],
            },
        },
    },
    # Emotion + LPP main effects (no interaction)
    {
        "model_name": "EEGMainEffects",
        "make_factory_path": "jaxcmr.models_eeg.eeg_cmr.make_factory",
        "parameters": {
            "fixed": {
                "allow_repeated_recalls": False,
                "modulate_emotion_by_primacy": False,
                "learn_after_context_update": False,
                "lpp_inter_scale": 0.0,
                "lpp_inter_threshold": 0.0,
            },
            "free": {
                "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "shared_support": [2.220446049250313e-16, 100.0],
                "item_support": [2.220446049250313e-16, 100.0],
                "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
                "primacy_scale": [2.220446049250313e-16, 100.0],
                "primacy_decay": [2.220446049250313e-16, 100.0],
                "choice_sensitivity": [2.220446049250313e-16, 100.0],
                "emotion_scale": [2.220446049250313e-16, 10.0],
                "lpp_main_scale": [2.220446049250313e-16, 100.0],
                "lpp_main_threshold": [-5.0, 5.0],
            },
        },
    },
    # Emotion + LPP main + interaction
    {
        "model_name": "EEGMainEffectsPlusInteraction",
        "make_factory_path": "jaxcmr.models_eeg.eeg_cmr.make_factory",
        "parameters": {
            "fixed": {
                "allow_repeated_recalls": False,
                "modulate_emotion_by_primacy": False,
                "learn_after_context_update": False,
            },
            "free": {
                "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "shared_support": [2.220446049250313e-16, 100.0],
                "item_support": [2.220446049250313e-16, 100.0],
                "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
                "primacy_scale": [2.220446049250313e-16, 100.0],
                "primacy_decay": [2.220446049250313e-16, 100.0],
                "choice_sensitivity": [2.220446049250313e-16, 100.0],
                "emotion_scale": [2.220446049250313e-16, 10.0],
                "lpp_main_scale": [2.220446049250313e-16, 100.0],
                "lpp_main_threshold": [-5.0, 5.0],
                "lpp_inter_scale": [2.220446049250313e-16, 100.0],
                "lpp_inter_threshold": [-5.0, 5.0],
            },
        },
    },
]

In [ ]:
root = find_project_root()
template = f"{root}/projects/TalmiEEG/notebooks/templates/cross_validation.ipynb"

for partial_params in varied_parameters:
    params = base_params.copy() | partial_params

    model_name = params["model_name"]
    run_tag = f"{params['base_run_tag']}_best_of_{params['best_of']}"

    output_path = (
        f"{root}/projects/TalmiEEG/notebooks/rendered/"
        f"cv_{params['data_tag']}_{model_name}_{run_tag}.ipynb"
    )
    print(f"Running CV for {model_name} -> {output_path}")

    pm.execute_notebook(template, output_path, parameters=params, progress_bar=True)

Running CV for EEGEmotionOnly -> /Users/jordangunn/jaxcmr/projects/TalmiEEG/notebooks/rendered/cv_TalmiEEG_EEGEmotionOnly_50_set_likelihood_fixed_term_best_of_3.ipynb


Executing:   0%|          | 0/8 [00:00<?, ?cell/s]

Running CV for EEGMainEffects -> /Users/jordangunn/jaxcmr/projects/TalmiEEG/notebooks/rendered/cv_TalmiEEG_EEGMainEffects_50_set_likelihood_fixed_term_best_of_3.ipynb


Executing:   0%|          | 0/8 [00:00<?, ?cell/s]

## Model Comparison

In [ ]:
# Load CV results for all models
run_tag = f"{base_params['base_run_tag']}_best_of_{base_params['best_of']}"
fits_dir = os.path.join(root, base_params["target_directory"], "fits")

cv_results = []
for partial_params in varied_parameters:
    model_name = partial_params["model_name"]
    cv_path = os.path.join(fits_dir, f"{base_params['data_tag']}_{model_name}_{run_tag}_cv.json")
    with open(cv_path) as f:
        result = json.load(f)
    result["name"] = model_name
    cv_results.append(result)
    cv_fitness = np.array(result["cv_fitness"])
    print(f"{model_name}: mean CV-NLL = {np.mean(cv_fitness):.2f} "
          f"(+/- {np.std(cv_fitness)/np.sqrt(len(cv_fitness))*1.96:.2f})")

In [ ]:
# Pairwise CV-NLL differences
mean_delta, ci_hw = pairwise_cv_differences(cv_results)

# Format as mean [95% CI]
n = len(cv_results)
names = [r["name"] for r in cv_results]
print("Pairwise CV-NLL differences (row - column):")
print("Negative values favor the row model.\n")
for i in range(n):
    for j in range(n):
        if i != j:
            m, c = mean_delta.iloc[i, j], ci_hw.iloc[i, j]
            print(f"  {names[i]} vs {names[j]}: {m:.2f} [{m-c:.2f}, {m+c:.2f}]")

In [ ]:
# Winner matrix: fraction of subjects where row model has lower CV-NLL
print("\nFraction of subjects where row model wins:")
print(cv_winner_comparison_matrix(cv_results).to_string(float_format="{:.2f}".format))